# Device Provisioning

Provision an IoT device inline from a notebook cell using `IAMContext`, then authenticate and send a telemetry sample. The device is automatically deactivated when the `async with` block exits.

In [ ]:
from datetime import datetime, timezone

from iam_sdk import IamClient, TelemetryRecord
from iam_sdk.jupyter import IAMContext

BASE_URL = "https://localhost:5161"
TENANT_ID = "<building-tenant-id>"

async with IamClient(BASE_URL) as auth:
    login = await auth.login("admin@example.com", "Password123!")

login.access_token[:12] + "..."

In [ ]:
async with IAMContext(BASE_URL, login.access_token, tenant_id=TENANT_ID) as ctx:
    registration, device = await ctx.provision_device(
        "notebook-sensor-001",
        device_type="sensor",
        authentication_method="hmac",
        permissions=["telemetry.publish"],
    )
    print(f"Provisioned {registration.device_id}, authenticated: {device.is_authenticated}")

    await device.send_telemetry([
        TelemetryRecord(
            device_id="notebook-sensor-001",
            metric_name="temperature",
            numeric_value=21.4,
            unit="celsius",
            timestamp=datetime.now(timezone.utc),
        )
    ])
    print("Telemetry sent")

# `notebook-sensor-001` is deactivated automatically here.